# Week 6 Lab 01: Confusion Matrix and Core Metrics with scikit-learn

**Scenario:** Cordwell Home and Hardware support triage
**Estimated duration:** 120 minutes
**Week 6 theme:** Model Evaluation Metrics and Experiment Tracking

Cordwell Home and Hardware routes incoming customer messages to one of two queues:

- `installation_issue` (label 1, the positive class): the customer reports a problem with a completed installation and needs a service visit.
- `general_inquiry` (label 0, the negative class): store hours, pricing, stock, policies, or questions about installation services that do not report a problem.

A missed installation issue is an unhappy customer with a real defect sitting in the wrong queue. A false alarm wastes a service coordinator's time. This lab is about measuring exactly those two failure modes.

## Objectives

By the end of this lab you will be able to:

1. Split a labeled dataset into stratified train, validation, and test sets and explain the role of each.
2. Train a TF-IDF plus Logistic Regression text classifier inside a scikit-learn Pipeline.
3. Construct and read a confusion matrix, naming TP, TN, FP, and FN in business terms.
4. Compute and interpret accuracy, precision, recall, and F1.
5. Show how moving the decision threshold trades precision against recall.
6. Demonstrate why accuracy is misleading on imbalanced data and why F1 is often the better headline number.

**Instructor solution notebook.** Fully implemented and executed. Withhold until after the lab.


## How this lab works

- Cells marked **PROVIDED** are plumbing. Run them and move on; they are not the lesson.
- Cells marked **TASK** contain a function contract and a `raise NotImplementedError`. Replace the raise with your implementation.
- Every task is followed by an **apply** cell that runs your function and shows its output, and a **checks** cell that scores it. The `check` harness never crashes the notebook: unimplemented tasks print `[TODO]` and failing checks print `[FAIL]`.
- Run All on a fresh copy of this notebook completes without errors and reports a low score. Your job is to raise the score to all checks passing.
- Two hint files ship with this lab. Pick one tier per task; reading both wastes time.
  - `HINTS.md`: three escalating levels per task. Start here if you want to be nudged, not carried.
  - `HINTS_DETAILED.md`: the working core of each task with line by line commentary. Start here if you are stuck or short on time.
- The target outputs you are aiming for appear in the **Worked target output** section below, before any task. You are coding toward a visible target, not reverse engineering assertions.


## Part 0: Setup

This lab runs entirely in-process: no Docker services, no local LLM server, and no network calls are needed. The MLflow tracking server and TruLens arrive later this week; today is pure scikit-learn.

Install the pinned dependencies once per environment:

```bash
pip install -r requirements.txt
```

Everything below runs on CPU in well under a minute on the cohort MacBooks. No device selection is needed because scikit-learn is CPU only.


In [ ]:
%pip install -r requirements.txt

In [ ]:
# PROVIDED: imports and global configuration

import random
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

# One seed for everything: corpus generation, data splits, and the model.
# Chosen so the trained classifier makes a realistic mix of both error types,
# which is exactly what a lab about confusion matrices needs.
RANDOM_SEED = 7

print(f"scikit-learn {sklearn.__version__}")
print(f"pandas       {pd.__version__}")
print(f"numpy        {np.__version__}")

In [ ]:
# PROVIDED: check harness
# check(name, fn) runs a zero-argument callable that returns True or False.
# It never raises: unimplemented tasks report [TODO], real failures report [FAIL].
# attempt(label, fn) runs your task functions the same way and returns the result,
# or None if the task is not implemented yet.

CHECK_RESULTS = {}

def check(name: str, fn) -> None:
    try:
        ok = bool(fn())
    except NotImplementedError:
        CHECK_RESULTS[name] = False
        print(f"[TODO] {name}: task not implemented yet")
        return
    except Exception as exc:
        CHECK_RESULTS[name] = False
        if isinstance(exc, NameError) or "NoneType" in str(exc):
            print(f"[TODO] {name}: depends on an earlier task")
        else:
            print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")
        return
    CHECK_RESULTS[name] = ok
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")

def attempt(label: str, fn):
    try:
        return fn()
    except NotImplementedError:
        print(f"[TODO] {label}: implement the task above, then re-run this cell.")
        return None
    except Exception as exc:
        print(f"[ERROR] {label}: {type(exc).__name__}: {exc}")
        return None

def summary() -> None:
    total = len(CHECK_RESULTS)
    passed = sum(CHECK_RESULTS.values())
    print(f"Checks passing: {passed}/{total}")

print("Check harness loaded.")

In [ ]:
# PROVIDED: environment checks (these should pass on a cold Run All)

check("env: scikit-learn 1.6 or newer", lambda: tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 6))
check("env: pandas 2.x or newer", lambda: int(pd.__version__.split(".")[0]) >= 2)
check("env: numpy 2.x or newer", lambda: int(np.__version__.split(".")[0]) >= 2)
summary()

## Part 1: The Cordwell support corpus (PROVIDED)

We generate a synthetic but domain realistic corpus of 500 Cordwell customer messages. Each message is an opening line, three to four body sentences drawn from template pools, and a closing line.

The generator deliberately produces three kinds of messages:

- **clear** (about 60 percent): unambiguous installation problems or unambiguous general inquiries.
- **hard** (about 20 percent): the label is correct but the wording is subtle. A hard positive hedges ("something seems a bit off"); a hard negative asks detailed questions about installation services without reporting any problem.
- **borderline** (about 20 percent): genuinely mixed signals, the kind of message human labelers disagree about. A customer asks about installation pricing and also mentions that one corner does not quite line up.

This matters for evaluation. A corpus of perfectly separable templates gives you a perfect classifier, an all diagonal confusion matrix, and nothing to learn. Real support queues contain ambiguity, so this one does too. Expect your model to make mistakes; the mistakes are the curriculum.

The generator is plumbing, not the lesson. Read it if you are curious, then run it and move on.


In [ ]:
# PROVIDED: sentence template pools for the Cordwell corpus generator

OPENINGS = [
    "Hi Cordwell team,",
    "Hello Cordwell support,",
    "Good afternoon,",
    "Hi there,",
    "Dear Cordwell customer service,",
    "Hey Cordwell,",
]

CLOSINGS = [
    "Thanks in advance for your help.",
    "I appreciate any guidance you can provide.",
    "Please let me know the next steps.",
    "Looking forward to your response.",
    "Thank you for your time.",
    "Hope to hear back soon.",
]

# Strong positive signal: an installation clearly went wrong.
INSTALL_PROBLEM = [
    "The vinyl plank flooring your crew installed last week is already lifting at the seams near the kitchen.",
    "After the cabinet installation, two of the doors will not close and one hinge is visibly bent.",
    "The dishwasher your installer connected is leaking water under the sink every time it runs.",
    "The ceiling fan that was mounted on Tuesday wobbles badly on the medium and high settings.",
    "Several tiles in the new backsplash cracked within days of the install being finished.",
    "The water heater replacement was completed yesterday but now there is no hot water at all.",
    "The storm door that was hung last month no longer latches and swings open in the wind.",
    "Grout is crumbling out of the joints in the shower your team retiled three weeks ago.",
    "The garage door opener stopped responding two days after your technician wired it in.",
    "The laminate countertop was installed with a visible gap where it meets the wall.",
    "The toilet your plumber set is rocking on the floor and the base seal appears to be failing.",
    "Paint is bubbling along the trim your painting crew finished in the hallway.",
]

# Medium positive signal: asking for a fix, warranty service, or a return visit.
INSTALL_FOLLOWUP = [
    "I would like someone to come back out and take a look as soon as possible.",
    "Can you schedule a repair visit under the installation warranty?",
    "This was supposed to be covered by the workmanship guarantee.",
    "I have photos of the damage if that helps your team assess it.",
    "The original work order number is on my receipt if you need it.",
    "We paid for professional installation and expected better results.",
]

# Strong negative signal: ordinary retail questions.
GENERAL_INQ = [
    "What time does the Powell location open on Sunday mornings?",
    "Do you carry the 18 volt cordless drill combo kit in stores or is it online only?",
    "I am comparing prices on interior paint and wondered if you match competitor coupons.",
    "Could you tell me whether the spring lawn sale includes riding mowers?",
    "Is there a military discount and does it apply to clearance items?",
    "How long do I have to return an unopened box of tile if I bought too much?",
    "Do you sell gift cards in custom amounts or only fixed denominations?",
    "Can I order lumber online and pick it up at the contractor desk the same day?",
    "What brands of smart thermostats do you stock at the moment?",
    "Does your rental center carry pressure washers on weekends?",
    "I am looking for a paint color match service and wondered what it costs.",
    "Are the holiday hours different for the garden center than the main store?",
]

# Ambiguous: installation vocabulary with no problem reported. Appears in both classes.
AMBIGUOUS = [
    "I had new flooring installed by your team earlier this year.",
    "We are planning a kitchen remodel and the cabinets were installed recently.",
    "Your installation crew was scheduled through the Powell store.",
    "The install itself happened about two weeks ago.",
    "I used your professional installation service for the appliances.",
    "We booked the measurement and installation package online.",
    "The project involved tile, grout, and a full backsplash install.",
    "Your installer left some paperwork about the warranty coverage.",
    "I have used Cordwell installation services twice before.",
    "The installation appointment went through your scheduling line.",
]

# Negative class, but saturated with installation vocabulary: questions about the service itself.
INSTALL_INQUIRY = [
    "How much does professional installation cost for a standard dishwasher?",
    "Does the installation quote include haul away of the old appliance?",
    "What is the typical lead time to schedule a flooring installation?",
    "Do your installers handle permits or is that on the homeowner?",
    "Is installation free if the appliance is over a certain price?",
    "Can I bundle installation for a washer and dryer purchased together?",
    "Does the installation warranty transfer if I sell the house?",
    "Are your installation crews employees or subcontractors?",
]

# Positive class, but soft and hedged wording.
MILD_PROBLEM = [
    "Something seems a bit off with how the door sits in the frame since the visit.",
    "I am not sure if this is normal but the finish looks uneven in a few spots.",
    "There is a faint noise coming from the unit that was not there before.",
    "One corner does not quite line up the way I expected it to.",
    "It might be settling but the gap seems to be getting wider.",
    "The surface feels slightly loose when I press on it near the edge.",
]

print(f"Template pools loaded: {len(INSTALL_PROBLEM)} problem, {len(GENERAL_INQ)} inquiry, {len(AMBIGUOUS)} ambiguous sentences.")

In [ ]:
# PROVIDED: corpus generator

def make_doc(rng: random.Random, label: int, kind: str) -> str:
    """Assemble one synthetic customer message of the given class and difficulty."""
    parts = [rng.choice(OPENINGS)]
    if kind == "clear" and label == 1:
        body = rng.sample(INSTALL_PROBLEM, 2) + rng.sample(INSTALL_FOLLOWUP, 1) + rng.sample(AMBIGUOUS, 1)
    elif kind == "clear" and label == 0:
        body = rng.sample(GENERAL_INQ, 3) + rng.sample(AMBIGUOUS, 1)
    elif kind == "hard" and label == 1:
        body = rng.sample(AMBIGUOUS, 2) + rng.sample(MILD_PROBLEM, 1) + rng.sample(INSTALL_FOLLOWUP, 1)
    elif kind == "hard" and label == 0:
        body = rng.sample(AMBIGUOUS, 2) + rng.sample(INSTALL_INQUIRY, 2)
    else:
        # Borderline: mixed signals in two flavors, so the overlap runs in both directions.
        if rng.random() < 0.5:
            body = (rng.sample(AMBIGUOUS, 1) + rng.sample(INSTALL_INQUIRY, 1)
                    + rng.sample(MILD_PROBLEM, 2))
        else:
            body = (rng.sample(AMBIGUOUS, 1) + rng.sample(INSTALL_INQUIRY, 1)
                    + rng.sample(MILD_PROBLEM, 1) + rng.sample(GENERAL_INQ, 1))
    rng.shuffle(body)
    parts.extend(body)
    parts.append(rng.choice(CLOSINGS))
    return " ".join(parts)


def build_corpus(n_docs: int = 500,
                 positive_fraction: float = 0.5,
                 seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Build a labeled Cordwell support corpus.

    Returns a DataFrame with columns:
      text        the full message
      label       1 for installation_issue, 0 for general_inquiry
      label_name  human readable class name
      kind        clear, hard, or borderline (for inspection only; never a feature)
    """
    rng = random.Random(seed)
    n_pos = int(round(n_docs * positive_fraction))
    n_neg = n_docs - n_pos
    rows = []
    for label, count in [(1, n_pos), (0, n_neg)]:
        for _ in range(count):
            r = rng.random()
            kind = "borderline" if r < 0.20 else ("hard" if r < 0.40 else "clear")
            rows.append({
                "text": make_doc(rng, label, kind),
                "label": label,
                "label_name": "installation_issue" if label == 1 else "general_inquiry",
                "kind": kind,
            })
    rng.shuffle(rows)
    return pd.DataFrame(rows).reset_index(drop=True)


corpus_df = build_corpus()
print(f"Corpus size: {len(corpus_df)}")
print()
print("Class distribution (percent):")
print(corpus_df["label_name"].value_counts(normalize=True).mul(100).round(1))
print()
print("Difficulty mix (percent):")
print(corpus_df["kind"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# PROVIDED: read two example messages, one from each class

for label_name in ["installation_issue", "general_inquiry"]:
    example = corpus_df[corpus_df["label_name"] == label_name].iloc[0]
    print(f"--- {label_name} ({example['kind']}) ---")
    print(example["text"])
    print()

# And one borderline message, the kind that creates honest classifier errors:
borderline = corpus_df[corpus_df["kind"] == "borderline"].iloc[0]
print(f"--- borderline example (labeled {borderline['label_name']}) ---")
print(borderline["text"])

In [ ]:
# PROVIDED: corpus checks (these should pass on a cold Run All)

check("corpus: 500 documents", lambda: len(corpus_df) == 500)
check("corpus: balanced classes", lambda: corpus_df["label"].mean() == 0.5)
check("corpus: expected columns", lambda: list(corpus_df.columns) == ["text", "label", "label_name", "kind"])
summary()

## Worked target output

Before you write any code, here is what done looks like. These are the actual numbers this notebook produces with `RANDOM_SEED = 7` on the pinned stack. Your outputs should match them exactly. Seeing the target does not solve the tasks; it tells you when you have solved them.

**After Task 1** (split sizes): train 350, validation 75, test 75, each split close to 50 percent positive.

**After Task 3** (validation metrics):

```
accuracy   0.867
precision  0.865
recall     0.865
f1         0.865
```

**After Task 4** (test confusion matrix, rows are true class, columns are predicted class, order 0 then 1):

```
[[33  4]
 [ 1 37]]
```

That is TN 33, FP 4, FN 1, TP 37, and test metrics of accuracy 0.933, precision 0.902, recall 0.974, F1 0.937.

**After Task 5** (threshold sweep on the test set):

```
 threshold  precision  recall     f1
       0.3      0.804   0.974  0.881
       0.5      0.902   0.974  0.937
       0.7      0.939   0.816  0.873
```

**After Tasks 6 and 7** (imbalanced corpus, 12 percent positive): the majority baseline scores accuracy 0.880 with recall 0.0 and F1 0.0. Logistic Regression scores accuracy 0.907 but recall 0.222: it looks fine by accuracy while missing 7 of the 9 real installation issues in the test set.


## Task 1: Train, validation, and test split (15 min)

Split the corpus 70-15-15 into train, validation, and test sets, stratified by label.

Why three sets and not two? The training set fits the model. The validation set is where you look while you tune: thresholds, hyperparameters, feature choices. The test set stays in a locked drawer until the very end, so the final number you report has never influenced a decision. If you tune against the test set, you are quietly fitting to it, and your reported metric becomes an advertisement instead of a measurement.

Why stratified? A random split of a small dataset can drift away from a 50-50 class balance by chance. `stratify=` forces every split to mirror the label distribution of the data being split, so metric differences between splits reflect the model, not sampling luck.

Mechanics: `train_test_split` only cuts two ways, so cut twice. First carve off 30 percent as a temporary pool, then cut that pool in half. Stratify both cuts and pass `random_state=seed` to both.


In [ ]:
# TASK 1: implement split_corpus

def split_corpus(df: pd.DataFrame, seed: int) -> Tuple[np.ndarray, ...]:
    """Split the corpus into stratified train (70%), validation (15%), and test (15%) sets.

    Args:
        df: corpus DataFrame with at least the columns "text" and "label".
        seed: random_state for both calls to train_test_split.

    Returns:
        A 6-tuple of numpy arrays:
        (X_train, y_train, X_val, y_val, X_test, y_test)
        where X arrays contain message text and y arrays contain integer labels.

    Requirements:
        - Use train_test_split twice: first split off 30% as a temporary pool,
          then split that pool 50-50 into validation and test.
        - Stratify the first cut by the full label array and the second cut
          by the temporary pool's label array.
        - Pass random_state=seed to both calls.
    """
    X = df["text"].to_numpy()
    y = df["label"].to_numpy()

    # First cut: 70% train, 30% temporary pool, stratified on the full labels.
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=seed
    )

    # Second cut: split the 30% pool in half, stratified on the pool's labels.
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# PROVIDED: apply Task 1 and inspect the splits

def show_class_distribution(name: str, labels: np.ndarray) -> None:
    pct = pd.Series(labels).value_counts(normalize=True).mul(100).round(1)
    print(f"{name}: n={len(labels)}, positive={pct.get(1, 0.0)}%")

split_result = attempt("Task 1 split_corpus", lambda: split_corpus(corpus_df, RANDOM_SEED))
if split_result is not None:
    X_train, y_train, X_val, y_val, X_test, y_test = split_result
    show_class_distribution("Train     ", y_train)
    show_class_distribution("Validation", y_val)
    show_class_distribution("Test      ", y_test)

In [ ]:
# PROVIDED: Task 1 checks

check("task1: split sizes are 350, 75, 75",
      lambda: (len(X_train), len(X_val), len(X_test)) == (350, 75, 75))
check("task1: no documents lost",
      lambda: len(X_train) + len(X_val) + len(X_test) == len(corpus_df))
check("task1: every split is stratified near 50 percent positive",
      lambda: all(0.45 <= np.mean(yy) <= 0.55 for yy in (y_train, y_val, y_test)))
summary()

## Task 2: Baseline pipeline, TF-IDF plus Logistic Regression (15 min)

Build and fit a scikit-learn `Pipeline` with two named steps:

- `"tfidf"`: a `TfidfVectorizer` with `ngram_range=(1, 2)`. TF-IDF turns each message into a vector of word and word pair weights: words that are frequent in this message but rare across the corpus score high. Bigrams let the model see that "not close" and "installation cost" mean different things than their individual words.
- `"clf"`: a `LogisticRegression` with `max_iter=1000` and `random_state=seed`. A linear classifier that outputs a probability per class. It is the standard first baseline for text classification: fast, strong, and easy to interpret.

Why a Pipeline instead of two separate objects? The vectorizer learns its vocabulary from the training data only. Wrap both steps in a Pipeline and that discipline is automatic: `fit` learns vocabulary and weights from training text, `predict` reuses the training vocabulary on new text. Keeping preprocessing and model in one object is the same habit that later makes experiment tracking and deployment sane, because there is exactly one artifact to version.


In [ ]:
# TASK 2: implement build_and_fit_pipeline

def build_and_fit_pipeline(X_train: np.ndarray, y_train: np.ndarray, seed: int) -> Pipeline:
    """Build a TF-IDF plus Logistic Regression pipeline and fit it on the training data.

    Args:
        X_train: array of message strings.
        y_train: array of integer labels.
        seed: random_state for LogisticRegression.

    Returns:
        A fitted sklearn Pipeline with steps named "tfidf" and "clf".

    Requirements:
        - TfidfVectorizer with ngram_range=(1, 2).
        - LogisticRegression with max_iter=1000 and random_state=seed.
        - Fit the pipeline on (X_train, y_train) before returning it.
    """
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

In [ ]:
# PROVIDED: apply Task 2

baseline_pipeline = attempt(
    "Task 2 build_and_fit_pipeline",
    lambda: build_and_fit_pipeline(X_train, y_train, RANDOM_SEED),
)
if baseline_pipeline is not None:
    vocab_size = len(baseline_pipeline.named_steps["tfidf"].vocabulary_)
    print(f"Pipeline fitted. TF-IDF vocabulary size: {vocab_size} unigrams and bigrams.")

In [ ]:
# PROVIDED: Task 2 checks

check("task2: pipeline has steps named tfidf and clf",
      lambda: set(baseline_pipeline.named_steps) == {"tfidf", "clf"})
check("task2: vectorizer uses unigrams and bigrams",
      lambda: baseline_pipeline.named_steps["tfidf"].ngram_range == (1, 2))
check("task2: pipeline is fitted and knows both classes",
      lambda: list(baseline_pipeline.named_steps["clf"].classes_) == [0, 1])
summary()

## Task 3: A reusable metrics function, checked on validation (10 min)

Write `compute_metrics(y_true, y_pred)` returning a dict with keys `accuracy`, `precision`, `recall`, and `f1`, each rounded to 3 decimals. You will reuse this function four times today, which is the point: an eval harness is unit testing for model behavior, and unit tests are functions, not copy pasted cells.

Plain-language reference for the four numbers, where "positive" means `installation_issue`:

- **Accuracy**: of all messages, what fraction did we route correctly?
- **Precision**: of the messages we flagged as installation issues, what fraction really were? Low precision means false alarms flooding the service coordinators.
- **Recall**: of the real installation issues, what fraction did we catch? Low recall means defects sitting in the wrong queue.
- **F1**: the harmonic mean of precision and recall. It only gets high when both are high, so it punishes the lazy tricks that inflate one at the expense of the other.

One sharp edge: pass `zero_division=0` to `precision_score`, `recall_score`, and `f1_score`. If a model never predicts the positive class, precision divides by zero; scikit-learn then warns and substitutes a value. Setting `zero_division=0` makes that substitution an explicit, silent 0.0. Task 6 relies on this: a baseline that never predicts positive should score precision 0, not raise a warning.

Apply it to the validation set first. The test set stays in the drawer.


In [ ]:
# TASK 3: implement compute_metrics

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute the four core classification metrics.

    Args:
        y_true: ground truth integer labels.
        y_pred: predicted integer labels.

    Returns:
        Dict with keys "accuracy", "precision", "recall", "f1",
        each a float rounded to 3 decimals.

    Requirements:
        - Use accuracy_score, precision_score, recall_score, f1_score.
        - Pass zero_division=0 to precision_score, recall_score, and f1_score.
    """
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 3),
    }

In [ ]:
# PROVIDED: apply Task 3 on the validation set

val_metrics = attempt(
    "Task 3 compute_metrics on validation",
    lambda: compute_metrics(y_val, baseline_pipeline.predict(X_val)),
)
if val_metrics is not None:
    for name, value in val_metrics.items():
        print(f"{name:<10} {value:.3f}")

In [ ]:
# PROVIDED: Task 3 checks
# The fixture check runs your function on a tiny hand-computable example:
# truth [1,1,1,0,0], prediction [1,0,1,1,0] gives TP=2, FN=1, FP=1, TN=1,
# so accuracy 0.6, precision 2 of 3, recall 2 of 3, f1 0.667.

_fixture_truth = np.array([1, 1, 1, 0, 0])
_fixture_pred = np.array([1, 0, 1, 1, 0])

check("task3: returns the four expected keys",
      lambda: set(compute_metrics(_fixture_truth, _fixture_pred)) == {"accuracy", "precision", "recall", "f1"})
check("task3: values are correct on a hand-computable fixture",
      lambda: compute_metrics(_fixture_truth, _fixture_pred) == {"accuracy": 0.6, "precision": 0.667, "recall": 0.667, "f1": 0.667})
check("task3: zero_division handled, all-negative predictions score 0 not a warning",
      lambda: compute_metrics(np.array([1, 0]), np.array([0, 0]))["precision"] == 0.0)
summary()

> **Reflection before moving on:**
> Validation precision and recall are both 0.865, so at the default threshold this model spreads its mistakes evenly between false alarms and misses. If Cordwell leadership says catching installation problems early matters more than a tidy queue, which of the two numbers would you push up first, and what would you expect to happen to the other one? Hold that thought for Task 5.


## Task 4: Confusion matrix and metrics on the held out test set (20 min)

Time to open the drawer. Generate predictions on the test set and build the confusion matrix.

The confusion matrix is the source of truth every metric is computed from. With `confusion_matrix(y_true, y_pred)` and classes 0 and 1, scikit-learn lays it out as:

```
                 predicted 0        predicted 1
true 0     TN (correct pass)    FP (false alarm)
true 1     FN (missed issue)    TP (caught issue)
```

In Cordwell terms: TN is a general inquiry routed normally, FP is a general inquiry that wrongly triggered a service visit, FN is a real installation defect sitting in the general queue, TP is a real defect correctly escalated.

Implement `predict_and_confusion(pipeline, X, y)` returning the tuple `(y_pred, cm)`. The plotting cell and the metrics readout below it are provided.


In [ ]:
# TASK 4: implement predict_and_confusion

def predict_and_confusion(pipeline: Pipeline, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Predict labels for X and build the confusion matrix against y.

    Args:
        pipeline: a fitted sklearn Pipeline.
        X: array of message strings.
        y: ground truth integer labels.

    Returns:
        (y_pred, cm) where y_pred is the predicted label array and
        cm is the 2x2 confusion matrix from confusion_matrix(y, y_pred).
    """
    y_pred = pipeline.predict(X)
    cm = confusion_matrix(y, y_pred)
    return y_pred, cm

In [ ]:
# PROVIDED: apply Task 4, plot the matrix, and print the test metrics

test_result = attempt(
    "Task 4 predict_and_confusion on test",
    lambda: predict_and_confusion(baseline_pipeline, X_test, y_test),
)
if test_result is not None:
    y_test_pred, cm = test_result
    print("Confusion matrix (rows true, columns predicted, order 0 then 1):")
    print(cm)
    print()

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["general_inquiry", "installation_issue"],
    )
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Cordwell triage classifier, test set")
    plt.tight_layout()
    plt.show()

    test_metrics = compute_metrics(y_test, y_test_pred)
    for name, value in test_metrics.items():
        print(f"{name:<10} {value:.3f}")
    print()
    print(classification_report(y_test, y_test_pred,
                                target_names=["general_inquiry", "installation_issue"]))

In [ ]:
# PROVIDED: Task 4 checks

check("task4: confusion matrix is 2x2 and covers all 75 test messages",
      lambda: cm.shape == (2, 2) and cm.sum() == 75)
check("task4: matrix is consistent with the returned predictions",
      lambda: bool(np.array_equal(cm, confusion_matrix(y_test, y_test_pred))))
check("task4: matrix matches the worked target output",
      lambda: bool(np.array_equal(cm, np.array([[33, 4], [1, 37]]))))
summary()

### Reading the matrix like an engineer

Answer these from the numbers on screen:

1. How many real installation issues did the model catch? That is TP, bottom right.
2. How many did it miss? That is FN, bottom left. Each one is a customer with a lifting floor or a leaking dishwasher waiting in the wrong queue.
3. How many false alarms did the coordinators absorb? That is FP, top right.
4. Test accuracy is 0.933 while validation accuracy was 0.867. Same model, same distribution. What explains the gap? Consider how much a metric computed on 75 messages can move from sampling alone, and keep that in mind whenever someone reports a single number without a sample size.


## Task 5: The decision threshold and the precision-recall trade (20 min)

`predict` is hiding a decision from you. Under the hood, `predict_proba` produces a probability that the message is an installation issue, and `predict` applies a fixed cutoff of 0.5. That cutoff is a business decision wearing a default's clothing.

- Lower the threshold and the model flags more messages: recall rises, precision falls. Right answer when missing a defect is expensive and coordinator time is cheap.
- Raise it and the model only speaks when confident: precision rises, recall falls. Right answer when the service team is small and every dispatch costs real money.

Implement `threshold_sweep(pipeline, X, y, thresholds)` returning a DataFrame with columns `threshold`, `precision`, `recall`, and `f1`, one row per threshold. Get the positive class probabilities from `predict_proba(X)[:, 1]`, then for each threshold convert probabilities to labels with a comparison and reuse `compute_metrics`.


In [ ]:
# TASK 5: implement threshold_sweep

def threshold_sweep(pipeline: Pipeline, X: np.ndarray, y: np.ndarray,
                    thresholds: Tuple[float, ...] = (0.3, 0.5, 0.7)) -> pd.DataFrame:
    """Evaluate precision, recall, and F1 at several decision thresholds.

    Args:
        pipeline: a fitted sklearn Pipeline supporting predict_proba.
        X: array of message strings.
        y: ground truth integer labels.
        thresholds: cutoffs to evaluate.

    Returns:
        DataFrame with columns ["threshold", "precision", "recall", "f1"],
        one row per threshold, in the given order.

    Requirements:
        - Compute positive class probabilities once with predict_proba(X)[:, 1].
        - A message is predicted positive when its probability is >= the threshold.
        - Reuse compute_metrics for the per-threshold numbers.
    """
    proba = pipeline.predict_proba(X)[:, 1]
    rows = []
    for threshold in thresholds:
        y_pred = (proba >= threshold).astype(int)
        metrics = compute_metrics(y, y_pred)
        rows.append({
            "threshold": threshold,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        })
    return pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])

In [ ]:
# PROVIDED: apply Task 5 on the test set

threshold_df = attempt(
    "Task 5 threshold_sweep",
    lambda: threshold_sweep(baseline_pipeline, X_test, y_test),
)
if threshold_df is not None:
    print(threshold_df.to_string(index=False))

In [ ]:
# PROVIDED: Task 5 checks

check("task5: DataFrame has the right shape and columns",
      lambda: list(threshold_df.columns) == ["threshold", "precision", "recall", "f1"] and len(threshold_df) == 3)
check("task5: recall never rises as the threshold rises",
      lambda: bool(np.all(np.diff(threshold_df["recall"].to_numpy()) <= 0)))
check("task5: precision at 0.7 is at least precision at 0.3",
      lambda: bool(threshold_df["precision"].iloc[2] >= threshold_df["precision"].iloc[0]))
summary()

### Which threshold would you ship?

From the table: 0.3 buys near total recall at the price of more false alarms, 0.7 buys precision at the price of missing roughly one issue in five, and 0.5 happens to give the best F1 here. There is no universally correct answer; there is a correct answer for a given cost of each error type. Be ready to defend a choice in the group discussion, in terms of coordinator hours and unhappy customers rather than decimals.

> **Currency note.** Slicing `predict_proba` by hand is the transparent way to learn this. In production code, current scikit-learn (1.5 and later) has first class tools for exactly this job: `FixedThresholdClassifier` wraps a model with your chosen cutoff so `predict` honors it everywhere downstream, and `TunedThresholdClassifierCV` picks the cutoff that maximizes a metric you name, using cross validation on the training data. The stretch section in the solution notebook shows both.


## Part 6: When accuracy lies, evaluation under class imbalance (25 min)

The balanced 50-50 corpus was a teaching convenience. In production, most Cordwell messages are not installation issues. We now rebuild the corpus with only 12 percent positives, which is closer to what the real queue looks like, and watch accuracy fall apart as a headline metric.

The provided cell below rebuilds the corpus and reuses your `split_corpus` from Task 1.


In [ ]:
# PROVIDED: build the imbalanced corpus (12 percent installation issues) and split it

imbalanced_df = build_corpus(n_docs=500, positive_fraction=0.12, seed=RANDOM_SEED)
print("Imbalanced class distribution (percent):")
print(imbalanced_df["label_name"].value_counts(normalize=True).mul(100).round(1))
print()

imb_split = attempt("split imbalanced corpus", lambda: split_corpus(imbalanced_df, RANDOM_SEED))
if imb_split is not None:
    X_train_imb, y_train_imb, X_val_imb, y_val_imb, X_test_imb, y_test_imb = imb_split
    show_class_distribution("Train     ", y_train_imb)
    show_class_distribution("Validation", y_val_imb)
    show_class_distribution("Test      ", y_test_imb)
    print(f"\nPositives in the imbalanced test set: {int(y_test_imb.sum())} of {len(y_test_imb)}")

## Task 6: The majority class baseline (10 min)

Before evaluating any real model on imbalanced data, compute the score of the dumbest possible competitor: a "model" that always predicts the most common training label. This is the sanity check baseline. Any model that cannot beat it is worse than a one line if statement.

Implement `majority_baseline_metrics(y_train, y_test)`:

1. Find the most frequent label in `y_train`.
2. Build a prediction array for the test set filled with that label.
3. Score it with your `compute_metrics` and return the dict, plus a `majority_label` key.


In [ ]:
# TASK 6: implement majority_baseline_metrics

def majority_baseline_metrics(y_train: np.ndarray, y_test: np.ndarray) -> dict:
    """Score an always-predict-the-majority-class baseline on the test set.

    Args:
        y_train: training labels, used only to find the majority class.
        y_test: test labels to score against.

    Returns:
        The compute_metrics dict plus one extra key, "majority_label",
        holding the majority class as a plain int.

    Hints in the contract:
        - np.bincount(y_train).argmax() finds the most frequent label.
        - np.full_like(y_test, fill_value) builds the constant prediction array.
    """
    majority_label = int(np.bincount(y_train).argmax())
    y_pred = np.full_like(y_test, majority_label)
    metrics = compute_metrics(y_test, y_pred)
    metrics["majority_label"] = majority_label
    return metrics

In [ ]:
# PROVIDED: apply Task 6

majority_metrics = attempt(
    "Task 6 majority_baseline_metrics",
    lambda: majority_baseline_metrics(y_train_imb, y_test_imb),
)
if majority_metrics is not None:
    print(f"Majority label: {majority_metrics['majority_label']} (general_inquiry)")
    for name in ["accuracy", "precision", "recall", "f1"]:
        print(f"{name:<10} {majority_metrics[name]:.3f}")

In [ ]:
# PROVIDED: Task 6 checks

check("task6: majority label is general_inquiry (0)",
      lambda: majority_metrics["majority_label"] == 0)
check("task6: accuracy is high despite predicting nothing useful",
      lambda: 0.80 <= majority_metrics["accuracy"] <= 0.95)
check("task6: recall and f1 are exactly zero",
      lambda: majority_metrics["recall"] == 0.0 and majority_metrics["f1"] == 0.0)
summary()

> **Pause on that accuracy number.** A model that helps no one just scored 0.880. If your team's dashboard showed only accuracy, this baseline would look production ready. Recall 0.0 is the honest number: it caught none of the customers who actually need help.


## Task 7: Logistic Regression on the imbalanced data (15 min)

Now train the real model on the imbalanced training set and put both models side by side. This is pure reuse of your earlier work: `build_and_fit_pipeline`, `predict_and_confusion`, and `compute_metrics`, pointed at the imbalanced splits.

Implement `train_and_evaluate(X_train, y_train, X_test, y_test, seed)` returning `(metrics_dict, cm)`.


In [ ]:
# TASK 7: implement train_and_evaluate

def train_and_evaluate(X_train: np.ndarray, y_train: np.ndarray,
                       X_test: np.ndarray, y_test: np.ndarray,
                       seed: int) -> Tuple[dict, np.ndarray]:
    """Train the standard pipeline on one dataset and score it on another.

    Args:
        X_train, y_train: training messages and labels.
        X_test, y_test: test messages and labels.
        seed: random_state passed through to the pipeline.

    Returns:
        (metrics, cm): the compute_metrics dict on the test set and
        the test confusion matrix.

    Requirements:
        - Reuse build_and_fit_pipeline, predict_and_confusion, and compute_metrics.
        - Do not duplicate any sklearn calls those functions already make.
    """
    pipeline = build_and_fit_pipeline(X_train, y_train, seed)
    y_pred, cm = predict_and_confusion(pipeline, X_test, y_test)
    metrics = compute_metrics(y_test, y_pred)
    return metrics, cm

In [ ]:
# PROVIDED: apply Task 7 and compare against the majority baseline

imb_result = attempt(
    "Task 7 train_and_evaluate on imbalanced data",
    lambda: train_and_evaluate(X_train_imb, y_train_imb, X_test_imb, y_test_imb, RANDOM_SEED),
)
if imb_result is not None:
    imb_metrics, cm_imb = imb_result

    comparison = pd.DataFrame([
        {"model": "majority baseline", **{k: majority_metrics[k] for k in ["accuracy", "precision", "recall", "f1"]}},
        {"model": "logistic regression", **imb_metrics},
    ])
    print(comparison.to_string(index=False))
    print()
    print("Imbalanced test confusion matrix (rows true, columns predicted):")
    print(cm_imb)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_imb,
        display_labels=["general_inquiry", "installation_issue"],
    )
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title("Imbalanced corpus (12 percent positive), test set")
    plt.tight_layout()
    plt.show()

In [ ]:
# PROVIDED: Task 7 checks

check("task7: model beats the baseline on F1",
      lambda: imb_metrics["f1"] > majority_metrics["f1"])
check("task7: accuracy gap between the models is small",
      lambda: abs(imb_metrics["accuracy"] - majority_metrics["accuracy"]) < 0.10)
check("task7: recall reveals most real issues are still missed",
      lambda: imb_metrics["recall"] < 0.5)
check("task7: confusion matrix accounts for all 75 test messages",
      lambda: cm_imb.sum() == 75)
summary()

## The punchline, and the recap

Look at the comparison table. By accuracy, the two models are 0.880 versus 0.907, nearly a tie. By recall, they are 0.0 versus 0.222: the trained model catches 2 of 9 real installation issues and misses 7. Accuracy compressed that catastrophe into a rounding difference. This is why accuracy is never the headline metric for rare but important events, and why F1, or a precision-recall pair chosen from business costs, is the number teams actually optimize.

In this lab you:

1. Built a realistic Cordwell support corpus with clear, hard, and borderline messages.
2. Split it into stratified train, validation, and test sets and used each for its proper job.
3. Trained a TF-IDF plus Logistic Regression pipeline as a strong text baseline.
4. Read a confusion matrix in business terms and computed the four core metrics from it.
5. Moved the decision threshold and watched precision and recall trade against each other.
6. Watched accuracy flatter a useless baseline under imbalance while recall and F1 told the truth.

**Where this goes next this week:** these metric functions become the scoring core of a real eval harness, we start logging every run to MLflow instead of scrolling notebook output, and we extend evaluation from classifiers to LLM outputs with TruLens.

### Stretch goals (attempt in any order)

Fully assembled stretch solutions live in the instructor solution notebook and are worked through after the lab.

1. **Fix the recall problem.** Retrain the imbalanced pipeline with `class_weight="balanced"` on the LogisticRegression and rebuild the comparison table. What happened to recall, precision, and accuracy? Which model would you actually deploy, and what did it cost you?
2. **Ship a threshold properly.** Use `FixedThresholdClassifier` with `FrozenEstimator` (both in current scikit-learn) to wrap the balanced-corpus pipeline with a 0.3 cutoff, so that plain `predict` honors your business decision. Confirm its confusion matrix matches your Task 5 row for 0.3.
3. **Let the data pick the cutoff.** Use `TunedThresholdClassifierCV(scoring="f1", cv=5)` to tune the threshold on the training data, then report the chosen `best_threshold_` and the resulting test metrics. Did it beat your hand picked 0.5?


## Stretch goal solutions (instructor notebook only)

These are worked in full below. Students receive the stretch prompts in the recap; these assembled solutions are withheld until after the lab.


In [ ]:
# STRETCH 1 SOLUTION: class_weight="balanced" on the imbalanced corpus
# class_weight="balanced" reweights the loss so each class contributes equally
# during training, even though positives are only 12 percent of the rows.
# The model stops treating rare positives as noise worth ignoring.

def build_and_fit_balanced_pipeline(X_train: np.ndarray, y_train: np.ndarray, seed: int) -> Pipeline:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=seed, class_weight="balanced")),
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

balanced_pipeline = build_and_fit_balanced_pipeline(X_train_imb, y_train_imb, RANDOM_SEED)
y_pred_bal, cm_bal = predict_and_confusion(balanced_pipeline, X_test_imb, y_test_imb)
balanced_metrics = compute_metrics(y_test_imb, y_pred_bal)

comparison3 = pd.DataFrame([
    {"model": "majority baseline", **{k: majority_metrics[k] for k in ["accuracy", "precision", "recall", "f1"]}},
    {"model": "logistic regression", **imb_metrics},
    {"model": "LR class_weight=balanced", **balanced_metrics},
])
print(comparison3.to_string(index=False))
print()
print("Balanced model confusion matrix (rows true, columns predicted):")
print(cm_bal)
print()
print("Reading: recall jumped from 0.222 to 1.000, every one of the 9 real issues is now")
print("caught, at the cost of 5 false alarms (precision 0.643). Accuracy moved only from")
print("0.907 to 0.933: it compressed a transformative fix into a couple of points, the")
print("same blindness it showed for the failure. The recall column tells the real story.")

In [ ]:
# STRETCH 2 SOLUTION: FixedThresholdClassifier with FrozenEstimator
# FrozenEstimator wraps the already fitted balanced-corpus pipeline so nothing refits.
# FixedThresholdClassifier then makes plain predict apply our 0.3 cutoff, so every
# downstream consumer (eval harness, API service, batch job) inherits the business
# decision without knowing about predict_proba.

from sklearn.model_selection import FixedThresholdClassifier
from sklearn.frozen import FrozenEstimator

low_threshold_model = FixedThresholdClassifier(
    FrozenEstimator(baseline_pipeline), threshold=0.3
)
# fit is required by the sklearn API contract; with FrozenEstimator it does not retrain.
low_threshold_model.fit(X_train, y_train)

y_pred_low = low_threshold_model.predict(X_test)
cm_low = confusion_matrix(y_test, y_pred_low)
low_metrics = compute_metrics(y_test, y_pred_low)

print("Confusion matrix at threshold 0.3 via FixedThresholdClassifier:")
print(cm_low)
print()
for name, value in low_metrics.items():
    print(f"{name:<10} {value:.3f}")
print()
row_03 = threshold_df[threshold_df["threshold"] == 0.3].iloc[0]
match = (low_metrics["precision"] == row_03["precision"]
         and low_metrics["recall"] == row_03["recall"]
         and low_metrics["f1"] == row_03["f1"])
print(f"Matches the Task 5 row for threshold 0.3: {match}")

In [ ]:
# STRETCH 3 SOLUTION: TunedThresholdClassifierCV
# Instead of hand picking a cutoff, let cross validation on the training data choose
# the threshold that maximizes F1. The test set still stays out of the decision.

from sklearn.model_selection import TunedThresholdClassifierCV

tuned_model = TunedThresholdClassifierCV(
    Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]),
    scoring="f1",
    cv=5,
)
tuned_model.fit(X_train, y_train)

y_pred_tuned = tuned_model.predict(X_test)
tuned_metrics = compute_metrics(y_test, y_pred_tuned)

print(f"Threshold chosen by cross validation: {tuned_model.best_threshold_:.3f}")
for name, value in tuned_metrics.items():
    print(f"{name:<10} {value:.3f}")
print()
print("Reading: cross validation chose a cutoff near 0.43 and landed at test F1 0.914,")
print("slightly below the 0.937 that the default 0.5 happened to achieve on this test set.")
print("That is not a failure of the tool. The tuned threshold was chosen on training data")
print("without peeking at the test set, which is exactly the discipline this lab teaches;")
print("the 0.5 result is one sample of test set luck, not a better methodology.")

## Final score

Run the cell below after finishing your tasks. Target: all checks passing.


In [ ]:
# PROVIDED: final score

summary()
if CHECK_RESULTS and all(CHECK_RESULTS.values()):
    print("All checks passing. Lab complete.")